In [1]:
from unsloth import FastLanguageModel
import torch

print("Loading LFM2.5-1.2B-Instruct from LiquidAI...")
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/LFM2.5-1.2B-Instruct",
    model_name="LiquidAI/LFM2.5-1.2B-Instruct",
    max_seq_length=8192,  # Choose any for long context!
    load_in_4bit=True,  # 4 bit quantization to reduce memory
    load_in_8bit=False,  # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit=False,  # [NEW!] Enables 16bit LoRA
    full_finetuning=False,  # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


D:\Cache\conda\envs\finetuner\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0316 22:42:55.378000 5220 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading LFM2.5-1.2B-Instruct from LiquidAI...
==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████| 148/148 [00:01<00:00, 79.18it/s, Materializing param=model.layers.15.operator_norm.weight]


Model loaded successfully!


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj",
                      "w1", "w2", "w3"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [47]:
import ast
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template, standardize_data_formats

# 1. Enforce a standard Chat Template on your tokenizer
# Unsloth automatically maps this to your specific base model's required format
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", 
)

### Load 2 datasets - Hinglish, English Pro

In [48]:
print("Loading gooftagoo dataset...")
# Gooftagoo has ~16.2k rows. We sample 10,000.
gooftagoo_ds = load_dataset("adi-kmt/gooftagoo", split="train")
gooftagoo_ds = gooftagoo_ds.select(range(10000))

print("Loading English anchor dataset...")
# FineTome acts as our logic anchor. We sample 2,000.
english_ds = load_dataset("mlabonne/FineTome-100k", split="train")
english_ds = english_ds.select(range(2000))

Loading gooftagoo dataset...
Loading English anchor dataset...


In [49]:
def format_gooftagoo(example):
    # Safely evaluate the string into a Python list
    try:
        raw_turns = ast.literal_eval(example["conversation"])
    except (ValueError, SyntaxError):
        raw_turns = []
        
    new_conversations = []
    # Break apart the {'user': '...', 'assistant': '...'} pairs into standard turns
    for turn in raw_turns:
        if "user" in turn:
            new_conversations.append({"role": "user", "content": turn["user"]})
        if "assistant" in turn:
            new_conversations.append({"role": "assistant", "content": turn["assistant"]})
            
    return {"conversations": new_conversations}

In [50]:
def format_english(example):
    new_conversations = []
    for msg in example["conversations"]:
        # Handle if it's already in role/content format
        if "role" in msg and "content" in msg:
            role = msg["role"]
            content = msg["content"]
        # Handle if it's in ShareGPT from/value format (just to be safe)
        elif "from" in msg and "value" in msg:
            role = msg["from"]
            if role == "human": role = "user"
            elif role == "gpt": role = "assistant"
            content = msg["value"]
        else:
            continue
            
        new_conversations.append({"role": role, "content": content})
        
    return {"conversations": new_conversations}

In [51]:
print("Standardizing Gooftagoo...")
gooftagoo_ds = gooftagoo_ds.map(format_gooftagoo)

print("Standardizing English dataset...")
english_ds = english_ds.map(format_english)

Standardizing Gooftagoo...
Standardizing English dataset...


In [52]:
gooftagoo_ds[0]

{'conversation': "[{'user': ' ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।', 'assistant': 'अरे, हाथ रगड़ने से गरम हो जाएंगे। लेकिन बता, तेरे हाथ अभी कैसे हैं? सूखे हैं, गीले हैं या कुछ लगा रखा है?'}, {'user': 'अभी तो सूखे हैं, लेकिन मैं सोच रहा था कि तेल या लोशन लगा लूं।', 'assistant': 'नहीं नहीं, अगर तुझे जल्दी से हाथ गरम करने हैं, तो सूखे हाथों से रगड़ना सबसे बेहतर होगा। गीले हाथों से या तेल, लोशन लगे हाथों से रगड़ने पर उतनी गर्मी नहीं मिलेगी।'}, {'user': 'अच्छा, सूखे हाथों से ही रगड़ना है। लेकिन ऐसा क्यों?', 'assistant': 'देख, जब हम सूखे हाथ रगड़ते हैं, तो घर्षण ज्यादा होता है, जिससे ज्यादा गर्मी पैदा होती है। गीले हाथों में या तेल, लोशन लगे हाथों में घर्षण कम होता है, इसलिए गर्मी कम मिलती है।'}]",
 'messages': [{'content': 'ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।',
   'role': 'user'},
  {'content': 'अरे, हाथ रगड़ने से गरम हो जाएंगे। लेकिन बता, तेरे हाथ अभी कैसे हैं? सूखे हैं, गीले हैं या कुछ लगा रखा है?',
   'role': 'assistant'},
  {'content': 'अभी तो सूख

In [53]:
english_ds[20]

{'conversations': [{'content': 'Write a program to find the number of letters in each word of a sentence using the map function.',
   'role': 'user'},
  {'content': 'Certainly! Here\'s a code example that uses the `map()` function to find the number of letters in each word of a sentence:\n\n```python\ndef count_letters(sentence):\n    """\n    This function takes a sentence as input and returns a list of the number of letters\n    in each word using the map function.\n\n    Args:\n        sentence (str): The input sentence containing words.\n\n    Returns:\n        list: A list of integers representing the number of letters in each word.\n\n    Examples:\n        >>> count_letters("Hello world")\n        [5, 5]\n        >>> count_letters("Python is awesome")\n        [6, 2, 7]\n    """\n    # Split the sentence into words\n    words = sentence.split()\n\n    # Use map to apply len() function on each word and get the length\n    letter_counts = list(map(len, words))\n\n    return letter

In [54]:
def unify_schema(example):
    standardized_turns = []

    # We check 'conversations' first, fallback to 'messages' if needed
    chat_data = example.get("conversations", example.get("messages", []))

    for msg in chat_data:
        # Catch both OpenAI (role/content) and ShareGPT (from/value) formats just in case
        role = msg.get("role", msg.get("from", ""))
        content = msg.get("content", msg.get("value", ""))

        # Standardize ShareGPT roles if they exist
        if role == "human": role = "user"
        if role == "gpt": role = "assistant"

        standardized_turns.append({
            "role": str(role),
            "content": str(content)
        })

    return {"conversations": standardized_turns}

In [55]:
gooftagoo_ds = gooftagoo_ds.map(unify_schema, remove_columns=gooftagoo_ds.column_names)
english_ds = english_ds.map(unify_schema, remove_columns=english_ds.column_names)

In [56]:
gooftagoo_ds[0]["conversations"]

[{'content': ' ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।',
  'role': 'user'},
 {'content': 'अरे, हाथ रगड़ने से गरम हो जाएंगे। लेकिन बता, तेरे हाथ अभी कैसे हैं? सूखे हैं, गीले हैं या कुछ लगा रखा है?',
  'role': 'assistant'},
 {'content': 'अभी तो सूखे हैं, लेकिन मैं सोच रहा था कि तेल या लोशन लगा लूं।',
  'role': 'user'},
 {'content': 'नहीं नहीं, अगर तुझे जल्दी से हाथ गरम करने हैं, तो सूखे हाथों से रगड़ना सबसे बेहतर होगा। गीले हाथों से या तेल, लोशन लगे हाथों से रगड़ने पर उतनी गर्मी नहीं मिलेगी।',
  'role': 'assistant'},
 {'content': 'अच्छा, सूखे हाथों से ही रगड़ना है। लेकिन ऐसा क्यों?',
  'role': 'user'},
 {'content': 'देख, जब हम सूखे हाथ रगड़ते हैं, तो घर्षण ज्यादा होता है, जिससे ज्यादा गर्मी पैदा होती है। गीले हाथों में या तेल, लोशन लगे हाथों में घर्षण कम होता है, इसलिए गर्मी कम मिलती है।',
  'role': 'assistant'}]

In [57]:
english_ds[20]["conversations"]

[{'content': 'Write a program to find the number of letters in each word of a sentence using the map function.',
  'role': 'user'},
 {'content': 'Certainly! Here\'s a code example that uses the `map()` function to find the number of letters in each word of a sentence:\n\n```python\ndef count_letters(sentence):\n    """\n    This function takes a sentence as input and returns a list of the number of letters\n    in each word using the map function.\n\n    Args:\n        sentence (str): The input sentence containing words.\n\n    Returns:\n        list: A list of integers representing the number of letters in each word.\n\n    Examples:\n        >>> count_letters("Hello world")\n        [5, 5]\n        >>> count_letters("Python is awesome")\n        [6, 2, 7]\n    """\n    # Split the sentence into words\n    words = sentence.split()\n\n    # Use map to apply len() function on each word and get the length\n    letter_counts = list(map(len, words))\n\n    return letter_counts\n```\n\nNow,

In [58]:
gooftagoo_ds.filter(lambda x: len(x["conversations"]) < 0)

Dataset({
    features: ['conversations'],
    num_rows: 0
})

In [59]:
english_ds.filter(lambda x: len(x["conversations"]) < 0)

Dataset({
    features: ['conversations'],
    num_rows: 0
})

In [60]:
mixed_dataset = concatenate_datasets([gooftagoo_ds, english_ds])
mixed_dataset = mixed_dataset.shuffle(seed=3407)

In [61]:
len(mixed_dataset)

12000

In [62]:
def formatting_prompts_func(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    # Remove BOS token if Unsloth/Tokenizer adds it automatically to prevent double-tokens
    return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

In [63]:
mixed_dataset = mixed_dataset.map(formatting_prompts_func, batched=True)

In [65]:
mixed_dataset[0]["text"]

"<|im_start|>user\n maine socha hai ki home improvement contractor ka kaam shuru karu. Tujhe kuch pata hai iske baare mein?<|im_end|>\n<|im_start|>assistant\nHaan, bilkul. Agar tu seriously is field mein aage badhna chahta hai, toh sabse pehle toh tuje kuch zaroori insurance aur licensing requirements ke baare mein pata hona chahiye. India mein, kai states mein contractors ko specific licenses aur insurance policies rakhne ki zarurat hoti hai taaki wo legally kaam kar sake.'<|im_end|>\n<|im_start|>user\nAchha, toh kis type ke insurance ki zarurat padti hai?<|im_end|>\n<|im_start|>assistant\nAchhi baat puchi. Mukhya roop se, do prakar ke insurance hote hain jo ek home improvement contractor ke liye zaroori hote hain: Liability Insurance aur Workers' Compensation Insurance. Liability Insurance kisi bhi property damage ya injuries ke liye coverage deta hai jo kaam ke dauraan ho sakti hai. Aur Workers' Compensation Insurance un workers ki protection ke liye hota hai jo injury ya illness ke

In [67]:
print("Splitting into Train and Eval sets...")
split_dataset = mixed_dataset.train_test_split(test_size=0.1, seed=3407)
train_data = split_dataset["train"]
eval_data = split_dataset["test"]

print(f"Final Training Rows: {len(train_data)} | Evaluation Rows: {len(eval_data)}")

Splitting into Train and Eval sets...
Final Training Rows: 10800 | Evaluation Rows: 1200


In [73]:
from trl import SFTTrainer, SFTConfig
# from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Initialize the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = eval_data, # We pass the eval set here so we can watch validation loss!
    # max_seq_length = 2048, # Capped at 2k context for training to protect 6GB VRAM
    # dataset_num_proc = 2,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, # Lowest possible batch size
        gradient_accumulation_steps = 8, # Simulates a batch size of 8

        # num_train_epochs = 1
        max_steps=60,

        # Evaluation Settings
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 1, # Prints validation loss at every step

        learning_rate = 2e-4,
        warmup_steps = 30,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "paged_adamw_8bit", # The VRAM safety net
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,

        output_dir = "outputs",
        report_to = "none", # Change to "wandb" if you set up a Weights & Biases account
    ),
)

Unsloth: Tokenizing ["text"]: 100%|████████████████████████████████████████| 1200/1200 [00:01<00:00, 786.42 examples/s]


In [74]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Filter: 100%|███████████████████████████████████████████████████████████| 10800/10800 [00:08<00:00, 1225.54 examples/s]


Unsloth: Removed 4 out of 10800 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Filter: 100%|█████████████████████████████████████████████████████████████| 1200/1200 [00:00<00:00, 1224.19 examples/s]


Unsloth: Removed 2 out of 1200 samples from eval_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [75]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 3050 6GB Laptop GPU. Max memory = 6.0 GB.
2.182 GB of memory reserved.


In [76]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,796 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 44,433,408 of 1,214,774,016 (3.66% trained)


Step,Training Loss,Validation Loss
1,2.708476,3.004655
2,3.196492,2.979378
3,3.380206,2.920493
4,2.266417,2.837911
5,2.207697,2.759780
6,3.159660,2.707840
7,2.485267,2.685879
8,2.422437,2.669812
9,2.584548,2.647897
10,2.366128,2.615885


Unsloth: Not an error, but Lfm2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [77]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

13859.1768 seconds used for training.
230.99 minutes used for training.
Peak reserved memory = 11.18 GB.
Peak reserved memory for training = 8.998 GB.
Peak reserved memory % of max memory = 186.333 %.
Peak reserved memory for training % of max memory = 149.967 %.


In [78]:
trainer_stats

TrainOutput(global_step=60, training_loss=2.1690770288308463, metrics={'train_runtime': 13859.1768, 'train_samples_per_second': 0.035, 'train_steps_per_second': 0.004, 'total_flos': 1629100687603200.0, 'train_loss': 2.1690770288308463, 'epoch': 0.0444609114486847})

In [ ]:
messages = [{
    "role": "user",
    "content": "ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 256,
    # Recommended Liquid settings!
    temperature = 0.1, top_k = 50, top_p = 0.1, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [83]:
messages = [{
    "role": "user",
    "content": "Find minima aur maximum 3x^2 - 5x^2 + 6",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 1024, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.5, top_k = 50, top_p = 0.1, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Let's break down the given expression:

$f(x) = 3x^2 - 5x^2 + 6$

First, we simplify the expression by combining like terms:

$f(x) = (3 - 5)x^2 + 6$
$f(x) = -2x^2 + 6$

Now, we need to find the minimum and maximum values of this function. Since the coefficient of $x^2$ is negative (-2), the parabola opens downward, which means it has a maximum value.

To find the maximum value, we need to find the vertex of the parabola. The x-coordinate of the vertex can be found using the formula:

$x = -\frac{b}{2a}$

In our case, $a = -2$ and $b = 0$. Plugging these values into the formula, we get:

$x = -\frac{0}{2(-2)}$
$x = 0$

Now that we have the x-coordinate of the vertex, we can find the corresponding y-value by plugging this value back into the original function:

$f(0) = -2(0)^2 + 6$
$f(0) = 6$

So, the maximum value of the function is 6, which occurs at $x = 0$.

To find the minimum value, we need to consider the behavior of the function as $x$ approaches positive or negative infinity. S

In [ ]:
messages = [{
    "role": "user",
    "content": "ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 1024, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.5, top_k = 50, top_p = 0.1, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

#### Kernel interrupted :sob:

In [2]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("Rescuing model from checkpoint...")

# 1. Load the model directly from your saved checkpoint folder!
checkpoint_path = "outputs/checkpoint-60" 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = checkpoint_path, 
    max_seq_length = 8192,
    dtype = None,
    load_in_4bit = True,
)

# 2. Re-apply the ChatML template so the tokenizer formats correctly
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

# 3. Fast inference setup (Prepares the model for exporting or chatting)
FastLanguageModel.for_inference(model)
print("Model loaded!")

Rescuing model from checkpoint...
==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████| 148/148 [00:01<00:00, 82.45it/s, Materializing param=model.layers.15.operator_norm.weight]


Model loaded!


In [2]:
ROOT_PATH = "D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct"
import os
os.makedirs(ROOT_PATH, exist_ok=True)

In [5]:
model.save_pretrained(f"{ROOT_PATH}/lora_r16_4bit_model")  # Local saving
tokenizer.save_pretrained(f"{ROOT_PATH}/lora_r16_4bit_model")

('D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/lora_r16_4bit_model\\tokenizer_config.json',
 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/lora_r16_4bit_model\\chat_template.jinja',
 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/lora_r16_4bit_model\\tokenizer.json')

In [3]:
if True:
    print("Let's load the saved LORA model")
    from unsloth import FastModel
    from transformers import Lfm2ForCausalLM
    print("Loading LORA r16 4 bit")
    model, tokenizer = FastModel.from_pretrained(
        model_name = f"{ROOT_PATH}/lora_r16_4bit_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 8192,
        # dtype = dtype,
        # auto_model = Lfm2ForCausalLM,
        load_in_4bit = True,
    )
    FastModel.for_inference(model) # Enable native 2x faster inference
print("Model Loaded successfully!")

Let's load the saved LORA model
Loading LORA r16 4 bit
==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████| 148/148 [00:01<00:00, 81.17it/s, Materializing param=model.layers.15.operator_norm.weight]


Model Loaded successfully!


In [ ]:
# messages = [{
#     "role": "user",
#     "content": "ठंड बहुत है, हाथ जल्दी से गरम करने का कोई तरीका बता ना।",
# }]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
#     tokenize = True,
#     return_dict = True,
# ).to("cuda")

# from transformers import TextStreamer
# _ = model.generate(
#     **inputs,
#     max_new_tokens = 256, # Increase for longer outputs!
#     # Recommended Liquid settings!
#     temperature = 0.5, top_k = 50, top_p = 0.9, repetition_penalty = 1.05,
#     streamer = TextStreamer(tokenizer, skip_prompt = True),
# )

In [4]:
if True:
    model.save_pretrained_gguf(
        f"{ROOT_PATH}/sft_multilingual_hindi",
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "f16"], 
    )

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: D:\Cache\hf\hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/sft_multilingual_hin


Successfully copied all 1 files from cache to `D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/sft_multilingual_hindi`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████████████| 1/1 [00:05<00:00,  5.27s/it]


Unsloth: Merge process complete. Saved to `D:\ML\Models\Finetuned\LiquidAI-LFM2.5-1.2B-Hindi-Instruct\sft_multilingual_hindi`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m', 'q8_0', 'f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/sft_multilingual_hindi_gguf\\LFM2.5-1.2B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth: [2] Converting GGUF bf16 into f16. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/sft_multilingual_hindi_gguf\\LFM2.5-1.2B-Instruct.F16.gguf', 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/sft_multilingual_hindi_gguf\\LFM2.5-1.2B-Instruct.Q8_0.gguf', 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Hindi-Instruct/sft_multilingual_hindi_gguf\\LFM2.5-1.2B-Instruct.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 

In [7]:
prompt = "bhai kaise ho aap"

# 3. Apply the ChatML template perfectly
messages = [
    {"role": "user", "content": prompt}
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # THIS adds the crucial <|im_start|>assistant\n trigger!
    return_tensors = "pt",
).to("cuda")

# 4. Generate the response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True) # skip_prompt hides the template tags from output

print("\n--- MODEL RESPONSE ---\n")
_ = model.generate(
    **inputs, 
    streamer = text_streamer, 
    max_new_tokens = 256, 
    use_cache = True,
    temperature = 0.4,
    repetition_penalty = 1.15,
    no_repeat_ngram_size = 3, #
)


--- MODEL RESPONSE ---

Dekh, bhai ke saath kuch samay mein apne aap ko shuruat kar sakte hain. Jaise ki agar tu apni aap se tareekein ya kisi baad mein badlavi toh, toh tu apne paas pehle apnaa jaayein aur unhe apne dhoondhan banane kiya jata hai.'<|im_end|>
